# Build Merged Instructions

This notebook merges prompt templates with actual data samples and saves the result locally as an Arrow dataset.

Each record contains the raw template, rendered instruction (split into input/output), and template metadata.

Use `push_merged_instructions.ipynb` to inspect and push the built dataset to HuggingFace.

In [1]:
import os
import requests
import string
from dotenv import load_dotenv
from tqdm.auto import tqdm

load_dotenv()

PROMPTLAB_API_URL = 'https://promptlab.up.railway.app/api/prompt/list?project_secret_key=6Wirj'
MAX_RETRIES = 10

prompts = None
for i in tqdm(range(MAX_RETRIES), desc="Fetching prompts"):
    api_response = requests.get(url=PROMPTLAB_API_URL)
    if api_response.ok:
        prompts = api_response.json()
        break

if not prompts:
    raise Exception('Failed to fetch prompts from PromptLab API')

print(f"Total prompts fetched: {len(prompts)}")

Fetching prompts:   0%|          | 0/10 [00:00<?, ?it/s]

Total prompts fetched: 365


In [2]:
filtered_prompts = [p for p in prompts if p['status'] == 'APPROVED']
print(f"Total approved prompts: {len(filtered_prompts)}")

# Prompter aliases: merge accounts belonging to the same person
PROMPTER_ALIASES = {
    "zaid": ["zaid", "zaid1"],
    "ahmed": ["ahmed", "ahmed6"],
    "irfan": ["irfan", "irfan9"],
}
prompter_alias_map = {v: canonical for canonical, variants in PROMPTER_ALIASES.items() for v in variants}

# Apply merging, then build anonymization mapping
for p in filtered_prompts:
    p['_merged_prompter'] = prompter_alias_map.get(p['created_by'], p['created_by'])
    p['tags'] = [tag for tag in p['tags'] if tag.strip()]

unique_prompters = sorted(set(p['_merged_prompter'] for p in filtered_prompts))
prompter_map = {name: string.ascii_lowercase[i] for i, name in enumerate(unique_prompters)}
print(f"Unique prompters (after merging): {len(unique_prompters)}")

Total approved prompts: 355
Unique prompters (after merging): 7


In [3]:
filtered_prompts[0]

{'id': 14898,
 'tags': ['Zero-shot COT'],
 'name': 'Prompt with zero-shot chain of thoughts',
 'task': {'name': 'claim verification'},
 'status': 'APPROVED',
 'template': "For the following task you have to label if the two sentences are of on of the following labels: {% for choice in answer_choices %}{{ choice }}{% if not loop.last %} or {% endif %}{% endfor %}. Sentence 1: {{s1}}\xa0 and sentence 2: {{s2}}.\r\nLet's think step by step:\r\n|||\r\n{{answer_choices[label]}}",
 'created_by': 'ahmed',
 'dataset_name': 'arbml/ANS_stance',
 'dataset_subset': 'default',
 'answer_choices': ['disagree', 'agree', 'other'],
 'text_direction': 'ltr',
 '_merged_prompter': 'ahmed'}

In [4]:
import datasets

# Anonymize prompters and remove status/created_by
for p in filtered_prompts:
    p['prompter_id'] = prompter_map[p['_merged_prompter']]

star_templates = datasets.Dataset.from_list(filtered_prompts).remove_columns(['status', 'created_by', '_merged_prompter'])
star_templates

Dataset({
    features: ['id', 'tags', 'name', 'task', 'template', 'dataset_name', 'dataset_subset', 'answer_choices', 'text_direction', 'prompter_id'],
    num_rows: 355
})

In [5]:
def clean_dataset_subset(subset):
    """Fix known dirty dataset_subset values from the API."""
    if subset.startswith('subtask5.arabic'):
        return 'subtask5.arabic'
    return subset

star_templates = star_templates.map(lambda example: {
    **example,
    'dataset_subset': clean_dataset_subset(example['dataset_subset']),
})

Map:   0%|          | 0/355 [00:00<?, ? examples/s]

In [6]:
from tqdm.auto import tqdm

all_datasets = {
    (prompt['dataset_name'], prompt['dataset_subset']) 
    for prompt in star_templates
}

def load_dataset_safe(dataset_name, dataset_subset):
    """Load a dataset, falling back to parquet export if scripts are unsupported."""
    try:
        return datasets.load_dataset(dataset_name, dataset_subset)
    except (RuntimeError, ValueError) as e:
        err_msg = str(e)
        if 'Dataset scripts are no longer supported' not in err_msg and "Couldn't find cache" not in err_msg:
            raise
        print(f'  -> {type(e).__name__}: falling back to parquet export for {dataset_name}')
        try:
            return datasets.load_dataset(dataset_name, dataset_subset, revision='refs/convert/parquet')
        except ValueError:
            print(f'  -> Config "{dataset_subset}" not found in parquet export, retrying with data_dir instead')
            return datasets.load_dataset(dataset_name, data_dir=dataset_subset, revision='refs/convert/parquet')

downloaded_datasets = {}
for dataset_name, dataset_subset in tqdm(all_datasets, desc="Downloading datasets"):
    print('extracting for dataset:', dataset_name, 'with subset:', dataset_subset)
    downloaded_datasets[(dataset_name, dataset_subset)] = load_dataset_safe(dataset_name, dataset_subset)

extracting for dataset: arbml/Arabic_RC_AQA with subset: default


extracting for dataset: arbml/Ashaar_dataset with subset: default


extracting for dataset: arbml/offenseval_2020 with subset: default


extracting for dataset: Goud/Goud-sum with subset: default


extracting for dataset: arbml/ACVA with subset: default


...

extracting for dataset: arbml/Commonsense_Validation with subset: default


extracting for dataset: arbml/MSAC with subset: default


extracting for dataset: arbml/arabic_text_diacritization with subset: default


extracting for dataset: arbml/APCD_remap with subset: default


extracting for dataset: arbml/Syria_Tweet_Sentiment with subset: default


In [7]:
from collections import defaultdict
splits_dict = defaultdict(list)
for dataset_name, dataset_details in downloaded_datasets.items():
    for split_name, split_data in dataset_details.items():
        splits_dict[f"{split_name}"].append(dataset_name)

In [8]:
# splits_dict

The output of the above cell should be:
```python
{
    'train': [('arbml/APCDv2', 'default'),
              ('FahdSeddik/AGS-Corpus', 'default'),
              ('arbml/ATT', 'default'),
              ('arbml/antcorpus', 'default'),
              ('arbml/PAAD', 'default'),
              ('arbml/OSAC_CNN', 'default'),
              ('arbml/MLMA_hate_speech_ar', 'default'),
              ('arbml/BBN_Blog_Posts', 'default'),
              ('Helsinki-NLP/opus-100', 'ar-en'),
              ('khalidalt/tydiqa-goldp', 'arabic'),
              ('arbml/Zero_Shot_Cross_Lingual_NER_ar_batched', 'default'),
              ('arbml/ElecMorocco', 'default'),
              ('arbml/Arabic_Hate_Speech', 'default'),
              ('arbml/arabic_text_diacritization', 'default'),
              ('arbml/ANS_stance', 'default'),
              ('ajgt_twitter_ar', 'plain_text'),
              ('arbml/AQMAR_batched', 'default'),
              ('sem_eval_2018_task_1', 'subtask5.arabic'),
              ('arbml/offenseval_2020', 'default'),
              ('arbml/Commonsense_Validation', 'default'),
              ('arbml/Ashaar_dataset', 'default'),
              ('arbml/Arabic_Dialects_Dataset', 'default'),
              ('arbml/Disease_NER_batched', 'default'),
              ('arbml/Twt15DA_Lists', 'default'),
              ('arbml/ArEntail', 'default'),
              ('arbml/Named_Entities_Lexicon', 'default'),
              ('arbml/shakkelha', 'default'),
              ('emotone_ar', 'default'),
              ('arbml/arastance', 'default'),
              ('arbml/Mawqif', 'default'),
              ('arbml/MPOLD', 'default'),
              ('arbml/caner_batched', 'default'),
              ('asas-ai/tydiqa-ar', 'secondary_task'),
              ('arbml/araData', 'default'),
              ('arbml/AT_ODSTA', 'default'),
              ('arbml/oclar', 'default'),
              ('Helsinki-NLP/opus_infopankki', 'ar-en'),
              ('wiki_lingua', 'arabic'),
              ('arbml/NileULex', 'default'),
              ('labr', 'plain_text'),
              ('hard', 'plain_text'),
              ('arbml/ArSarcasm_v2', 'default'),
              ('ar_sarcasm', 'default'),
              ('arbml/ArMATH', 'default'),
              ('arbml/qa4mre', 'default'),
              ('arbml/Dangerous_Dataset', 'default'),
              ('arbml/ArabicTE', 'default'),
              ('arbml/SANAD', 'default'),
              ('GEM/xlsum', 'arabic'),
              ('arbml/AraSum', 'default'),
              ('arbml/ArCovidVac', 'default'),
              ('arbml/APCD_remap', 'default'),
              ('mkqa', 'mkqa'),
              ('arbml/Shami', 'default')],
    'test': [('facebook/belebele', 'acm_Arab'),
              ('Helsinki-NLP/opus-100', 'ar-en'),
              ('OALL/Arabic_EXAMS', 'default'),
              ('arbml/arabic_text_diacritization', 'default'),
              ('arbml/ANS_stance', 'default'),
              ('sem_eval_2018_task_1', 'subtask5.arabic'),
              ('arbml/offenseval_2020', 'default'),
              ('arbml/nsurl_2019_task8_test', 'default'),
              ('arbml/ArEntail', 'default'),
              ('Helsinki-NLP/tatoeba_mt', 'ara-eng'),
              ('arbml/arastance', 'default'),
              ('arbml/ArabicMMLU', 'default'),
              ('arbml/ACVA', 'default'),
              ('labr', 'plain_text'),
              ('arbml/ArSarcasm_v2', 'default'),
              ('arbml/iSarcasmEval_task_A', 'default'),
              ('ar_sarcasm', 'default'),
              ('arbml/CIDAR-MCQ-100', 'default'),
              ('GEM/xlsum', 'arabic')],         
    'validation': [('Helsinki-NLP/opus-100', 'ar-en'),
              ('arbml/AraBench_dev', 'default'),
              ('khalidalt/tydiqa-goldp', 'arabic'),
              ('OALL/Arabic_EXAMS', 'default'),
              ('arbml/Arabic_Hate_Speech', 'default'),
              ('arbml/arabic_text_diacritization', 'default'),
              ('arbml/ANS_stance', 'default'),
              ('sem_eval_2018_task_1', 'subtask5.arabic'),
              ('arbml/Commonsense_Validation', 'default'),
              ('Helsinki-NLP/tatoeba_mt', 'ara-eng'),
              ('arbml/arastance', 'default'),
              ('asas-ai/tydiqa-ar', 'secondary_task'),
              ('arbml/ACVA', 'default'),
              ('xquad', 'xquad.ar'),
              ('GEM/xlsum', 'arabic')]
}
```

## Merge Templates with Data Samples

In [9]:
from jinja2 import Environment, StrictUndefined
from typing import Dict, List, Optional, Tuple, Any


class TemplateProcessor:

    def __init__(self):
        self.env = Environment(undefined=StrictUndefined)
    
    def apply_template(
        self,
        prompt_template: Dict[str, Any],
        sample: Dict[str, Any],
    ) -> Tuple[bool, str]:

        # Validate template has divider
        template_content = prompt_template['template']
        answer_choices = prompt_template.get('answer_choices', [])
        
        if "|||" not in template_content:
            return False, "Template must contain ||| divider"
        
        # Prepare sample with answer choices
        sample_with_choices = sample.copy()
        if answer_choices:
            sample_with_choices["answer_choices"] = answer_choices
        
        try:
            # Render the template
            template = self.env.from_string(template_content)
            rendered = template.render(**sample_with_choices)
            # Validate answer choices if provided
            if answer_choices:
                validation_error = self._validate_answer_choices(
                    rendered, answer_choices
                )
                if validation_error:
                    return False, validation_error
            
            return True, rendered
            
        except Exception as e:
            return False, f"Error rendering template: {str(e)}"
    
    def _validate_answer_choices(
        self, 
        rendered_template: str, 
        answer_choices: List[str]
    ) -> Optional[str]:
        # Extract the answer part (after |||)
        parts = rendered_template.split("|||")
        if len(parts) < 2:
            return "Template did not produce expected ||| division"
        
        answer_part = parts[-1].strip()
        
        # Check each comma-separated answer
        for answer in answer_part.split(","):
            answer_clean = answer.strip()
            if answer_clean and answer_clean not in answer_choices:
                return f"Output '{answer_clean}' is not in answer_choices {answer_choices}"
        return None

# Convenience function for simple use cases
def apply_template_to_sample(
    prompt: Dict[str, Any],
    sample: Dict[str, Any],
) -> str:
    processor = TemplateProcessor()
    success, result = processor.apply_template(prompt, sample)
    
    if not success:
        raise ValueError(result)
    
    return result

In [10]:
import json
import os
import gc
from collections import defaultdict

# --- Configuration ---
OUTPUT_DIR = 'Notebooks/push_to_hf/built_datasets/star-instructions'
BATCH_FILE = os.path.join(OUTPUT_DIR, '_build_batches.jsonl')
BATCH_SIZE = 256_000

os.makedirs(OUTPUT_DIR, exist_ok=True)
if os.path.exists(BATCH_FILE):
    os.remove(BATCH_FILE)

# --- Step 1: Group prompts by dataset ---
# Instead of iterating prompt-first (which requires all datasets in memory
# throughout), we iterate dataset-first so we can free each dataset after
# all its prompts have been processed.
prompts_by_dataset = defaultdict(list)
for prompt in star_templates:
    key = (prompt['dataset_name'], prompt['dataset_subset'])
    prompts_by_dataset[key].append(prompt)

# --- Step 2: Helper to flush a batch buffer to disk ---
def flush_batch(buffer, path):
    with open(path, 'a') as f:
        for record in buffer:
            f.write(json.dumps(record, ensure_ascii=False) + '\n')

# --- Step 3: Process one dataset at a time, stream results to JSONL ---
failed_prompts = []
bad_splits = []
total_written = 0
batch_buffer = []

for dataset_key, dataset_prompts in tqdm(prompts_by_dataset.items(), desc="Processing datasets"):
    dataset_splits = downloaded_datasets[dataset_key]

    for prompt in dataset_prompts:
        for split_name in dataset_splits.keys():
            for sample in tqdm(
                dataset_splits[split_name],
                desc=f"Applying template '{prompt['name'][:10]}...' on {split_name} split of {dataset_key}",
                leave=False,
            ):
                try:
                    merged_prompt = dict(prompt)
                    merged_prompt['split_name'] = split_name
                    merged_prompt['full_instruction'] = apply_template_to_sample(merged_prompt, sample)

                    prompt_template = merged_prompt.pop('template')
                    merged_prompt['instruction_template'] = prompt_template
                    merged_prompt['instruction_name'] = merged_prompt.pop('name')
                    merged_prompt['instruction_tasks'] = list(merged_prompt.pop('task').values())

                    splitted_instruction = merged_prompt['full_instruction'].split('|||')
                    if len(splitted_instruction) != 2:
                        bad_splits.append({
                            'prompt_id': merged_prompt['id'],
                            'prompt_name': merged_prompt.get('instruction_name', prompt['name']),
                            'dataset': dataset_key,
                            'split': split_name,
                            'num_parts': len(splitted_instruction),
                        })
                        continue

                    merged_prompt['instruction_input'] = splitted_instruction[0].strip()
                    merged_prompt['instruction_output'] = splitted_instruction[1].strip()
                    merged_prompt['instruction_template_id'] = merged_prompt.pop('id')
                    batch_buffer.append(merged_prompt)
                    total_written += 1

                    # Flush to disk periodically to avoid accumulating in RAM
                    if len(batch_buffer) >= BATCH_SIZE:
                        flush_batch(batch_buffer, BATCH_FILE)
                        batch_buffer.clear()

                except Exception as e:
                    failed_prompts.append({
                        'prompt_id': prompt['id'],
                        'prompt_name': prompt['name'],
                        'dataset': dataset_key,
                        'split': split_name,
                        'reason': str(e),
                    })

    # Free this dataset from memory now that all its prompts are done
    del downloaded_datasets[dataset_key]
    gc.collect()

# Flush any remaining records
if batch_buffer:
    flush_batch(batch_buffer, BATCH_FILE)
    batch_buffer.clear()

gc.collect()
print(f"Total merged instructions written to {BATCH_FILE}: {total_written}")
print(f"Bad ||| splits (data/template issue): {len(bad_splits)}")
print(f"Failed prompts (rendering errors): {len(failed_prompts)}")

Processing datasets:   0%|          | 0/87 [00:00<?, ?it/s]

Applying template 'Prompt wit...' on train split of ('arbml/ANS_stance', 'default'):   0%|          | 0/2652 […

Applying template 'Prompt wit...' on validation split of ('arbml/ANS_stance', 'default'):   0%|          | 0/7…

Applying template 'Prompt wit...' on test split of ('arbml/ANS_stance', 'default'):   0%|          | 0/379 [00…

Applying template 'Structured...' on train split of ('arbml/ANS_stance', 'default'):   0%|          | 0/2652 […

...

Applying template 'Sentiment ...' on train split of ('arbml/AT_ODSTA', 'default'):   0%|          | 0/3000 [00…

Applying template 'Sentiment ...' on train split of ('arbml/AT_ODSTA', 'default'):   0%|          | 0/3000 [00…

Applying template 'Sentiment ...' on train split of ('arbml/AT_ODSTA', 'default'):   0%|          | 0/3000 [00…

Applying template 'Sentiment ...' on train split of ('arbml/AT_ODSTA', 'default'):   0%|          | 0/3000 [00…

Applying template 'Example Pr...' on train split of ('arbml/AT_ODSTA', 'default'):   0%|          | 0/3000 [00…

In [ ]:
if bad_splits:
    unique_bad = {(b['prompt_id'], b['num_parts']): b for b in bad_splits}
    print(f"\n--- Bad ||| splits ({len(unique_bad)} unique prompt/part combos) ---")
    for b in unique_bad.values():
        print(f"  - Prompt {b['prompt_id']} ({b['prompt_name']}) on {b['dataset']} [{b['split']}]: got {b['num_parts']} parts instead of 2")

In [ ]:
if failed_prompts:
    unique_failures = {(f['prompt_id'], f['reason']): f for f in failed_prompts}
    print(f"\n--- Rendering errors ({len(unique_failures)} unique) ---")
    for f in unique_failures.values():
        print(f"  - Prompt {f['prompt_id']} ({f['prompt_name']}) on {f['dataset']} [{f['split']}]: {f['reason']}")

In [ ]:
# Peek at the first record from the JSONL file
with open(BATCH_FILE) as f:
    first_record = json.loads(f.readline())
first_record

## Save to Local Directory

In [ ]:
# Load the streamed JSONL into a HuggingFace Dataset and save as Arrow
star_dataset = datasets.Dataset.from_json(BATCH_FILE)
print(star_dataset)

star_dataset.save_to_disk(OUTPUT_DIR)
print(f"Saved {len(star_dataset)} merged instructions to {OUTPUT_DIR}")

# Clean up the intermediate JSONL file
os.remove(BATCH_FILE)
print(f"Removed intermediate file: {BATCH_FILE}")